In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

def bs_delta(S,K, tau, r_k, sigma):
    if tau > 0:
        d1 = (np.log(S/K) + (r_k + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
        return norm.cdf(d1)
    else:
        return 1 if S >= K else 0
        
def delta_hedging(price_segment_array_Sd, price_segment_array_Sdd, window, rebalance_freq, transaction_cost,r):
    ANNUAL_DAYS = 252
    transaction_cost = float(transaction_cost)
    r = float(r)
    prices_Sdd = price_segment_array_Sdd.astype(np.float64)
    prices_Sd = price_segment_array_Sd.astype(np.float64)
    T = window / ANNUAL_DAYS
    dt = 1/ANNUAL_DAYS
    alpha = 1
    
    vol_window_Sd = prices_Sd[:window]
    hedge_window_Sd = prices_Sd[window:window+(window+1)]
    hedge_window_Sdd = prices_Sdd[window:window+(window+1)]
    log_returns_Sd_raw = np.log(vol_window_Sd[1:] / vol_window_Sd[:-1])
    fresh_returns = log_returns_Sd_raw[log_returns_Sd_raw != 0.0]
    
    if len(fresh_returns) >1:
        sigma = np.std(fresh_returns) * np.sqrt(ANNUAL_DAYS)
    else:
        sigma = np.std(log_returns_Sd_raw)* np.sqrt(ANNUAL_DAYS)
    K = float(hedge_window_Sd[0])
    k_K = transaction_cost
    HEDGE_WINDOW_SIZE =window + 1
    rebalance_pending = False
    for t_idx in range(HEDGE_WINDOW_SIZE):
        St_Sd = float(hedge_window_Sd[t_idx])
        St_Sdd = float(hedge_window_Sdd[t_idx])
        tau = float((window - t_idx) / ANNUAL_DAYS)
        
        if t_idx == 0:
            k_K = transaction_cost
            delta_Sd_0 = bs_delta(St_Sd, K, T, r, sigma)
            cash = St_Sd * delta_Sd_0 * (1-k_K)
            delta_before_K = delta_Sd_0
        else:
            is_stale = (St_Sd == float(hedge_window_Sd[t_idx -1]))
            if t_idx % rebalance_freq == 0 and tau >0:
                rebalance_pending = True
            if tau >0:
                if rebalance_pending and not is_stale:
                    delta_Sd = bs_delta(St_Sd, K, tau, r, sigma)
                    cash = ((1 + r *dt)*cash) + (delta_Sd - delta_before_K)* St_Sd *(1-np.sign(delta_Sd -delta_before_K)*k_K)
                    delta_before_K = delta_Sd
                    rebalance_pending = False
                else:
                    cash = (1 + r *dt) * cash
            if tau ==0:
                S_T = alpha * St_Sdd + (1 - alpha) * (St_Sd)
                delta_Sd_final = bs_delta(St_Sd, K,0,r,sigma)
                final_cash_K = (1 + r *dt) * cash + (delta_Sd_final - delta_before_K) * St_Sd*(1-np.sign(delta_Sd_final - delta_before_K)*k_K) - (delta_Sd_final *St_Sd) + max(S_T - K, 0)
                final_cash = np.exp(-r * T) * final_cash_K
                return final_cash
                
def newey_west_se(x, lag):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    xm = x - x.mean()
    gamma_0 = np.dot(xm, xm) / n
    long_run_var = gamma_0
    max_lag = min(lag, n - 1)
    for k in range(1, max_lag + 1):
        gamma_k = np.dot(xm[k:], xm[:-k]) / n
        weight = 1 - k / (max_lag + 1)
        long_run_var += 2 * weight * gamma_k
    var_of_mean = long_run_var / n
    return np.sqrt(max(var_of_mean, 0.0))
    
csv_file_path = 'combined_data_EBL_KCBUG_KCBKY.csv'
price_column_Sdd = 'PX_LAST_EBL_file1'
price_column_Sd = 'PX_LAST_file3'
df = pd.read_csv(csv_file_path)
prices_array_Sd = df[price_column_Sd].values.astype(np.float64)
prices_array_Sdd = df[price_column_Sdd].values.astype(np.float64)

window = 84
hedge_window_size = window +1
risk_free_rate = 0.1
transaction_costs = [0.005,0.01,0.02]
rebalance_frequencies = [1,2,3,4,6,7,12,14,21,28,42,84]
step_size = 1
n_total = len(prices_array_Sd)
hedge_starts = list(range(window, n_total - hedge_window_size + 1, step_size))
num_segments = len(hedge_starts)

print(f"step_size = {step_size} days")
print(f"Total number of partially-overlapping hedge segments: {num_segments}")
print(f"(each hedge window = {hedge_window_size} days; volatility window = preceding {window} days)")

summary_data = []
for cost in transaction_costs:
    for freq in rebalance_frequencies:
        final_cashs = []
        for start in hedge_starts:
            vol_prices_Sd =prices_array_Sd[start - window:start]
            hedge_prices_Sd = prices_array_Sd[start:start +hedge_window_size]
            hedge_prices_Sdd = prices_array_Sdd[start:start +hedge_window_size]
            if len(vol_prices_Sd) < window or len(hedge_prices_Sd) < hedge_window_size:
                continue
            segment_array_Sd = np.concatenate([vol_prices_Sd,  hedge_prices_Sd])
            segment_array_Sdd = np.concatenate([vol_prices_Sd,  hedge_prices_Sdd])
            final_cash = delta_hedging(segment_array_Sd, segment_array_Sdd, window=window, rebalance_freq=freq,transaction_cost=cost,r=risk_free_rate)
            final_cashs.append(final_cash)
            
        final_cashs = np.array(final_cashs)
        n_segments = len(final_cashs)
        bias_val = np.mean(final_cashs)
        std_val = np.std(final_cashs)
        overlap_lag = max(1, hedge_window_size - step_size)
        nw_se_val = newey_west_se(final_cashs, lag=overlap_lag)
        
        print(f"\nTransaction cost: {cost*100:.2f}%, Rebalancing every {freq} day(s)")
        print(f"Number of segments counted: {n_segments}")
        print(f"Bias (mean of final_cash): {bias_val:.4f}")
        print(f"Std of final_cash: {std_val:.4f}")
        print(f"Newey-West SE of Bias (lag={overlap_lag}): {nw_se_val:.4f}")
        
        summary_data.append({'TransactionCost': cost, 'RebalanceFreq': freq, 'Bias': bias_val, 'Std': std_val, 'NW_SE_Bias': nw_se_val})

df_summary = pd.DataFrame(summary_data)
color_low_cost = 'red'
color_mid_cost = 'purple'
color_high_cost = 'green'
colors = [color_low_cost, color_mid_cost, color_high_cost]
linestyles = ['-', '--', '-.']
markers = ['o', 's', '^']
LABEL_FS,TICK_FS, LEGEND_FS = 20, 12,14
LINE_W, MARKER_S = 2.5, 10
def style_axes(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel, fontweight = 'bold', fontsize = LABEL_FS)
    ax.set_ylabel(ylabel, fontweight = 'bold', fontsize = LABEL_FS)
    ax.set_xticks(rebalance_frequencies)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')
        lab.set_fontsize(TICK_FS)
    ax.grid(True)
    leg = ax.legend(prop={'weight': 'bold', 'size': LEGEND_FS})
    for ln in leg.get_lines():
        ln.set_linewidth(LINE_W)
        ln.set_markersize(MARKER_S - 2)

def make_plot(metric, ylabel, with_error=False):
    fig, ax = plt.subplots(figsize=(12,6))
    for cost, ls, mk, c in zip(transaction_costs, linestyles, markers, colors):
        sub = df_summary[df_summary['TransactionCost'] == cost]
        label = f'Transaction Cost {cost*100:.2f}%'
        if with_error:
            ax.errorbar(sub['RebalanceFreq'], sub[metric],yerr=sub['NW_SE_Bias'], marker=mk, linestyle = ls, color=c,ecolor=c,
                       linewidth=LINE_W, markersize=MARKER_S,elinewidth=LINE_W,  capsize = 4, capthick = LINE_W, label = label)
        else:
            ax.plot(sub['RebalanceFreq'], sub[metric], marker=mk, linestyle=ls, color=c, linewidth = LINE_W, markersize=MARKER_S, label=label)
    style_axes(ax, 'Rebalancing interval (days)', ylabel)
    fig.tight_layout()
    return fig, ax

make_plot('Bias', 'Mean final cash (with N-W SE)', with_error=True)
plt.show()
make_plot('Std' , 'Std of final cash')
plt.show()
make_plot('NW_SE_Bias', 'N-W SE of the mean final cash')
plt.show
######
